<table><tr>
<td><b>Sapienza University of Rome</b><br>PhD Soft Skills 2026</td>
</tr></table>

# Notebook 2 — Images, and how to fool them
**5 minutes — watch this one.** Your instructor runs it on screen. You do not need to type anything — but you can, and it takes under a minute.


Everything so far has been text. Images are the other classic case, and they fail in a way that is worth seeing with your own eyes.

We use the handwritten-digit dataset that ships inside scikit-learn — 1,797 tiny images of numbers, each 8×8 pixels. No download, no GPU, no waiting.

### 1 · Build an image classifier

**Copy this into the AI box** (click the empty cell below, then `Ctrl`+`Shift`+`Enter`, or the **Generate** button):

```text
Load the built-in handwritten digits dataset from scikit-learn.
Show me 10 example images with their labels.
Then train a classifier on 80% of the data and report its
accuracy on the remaining 20%.
Finally show a grid of the images it got wrong, each labelled
with the true digit and what it guessed.
```

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
import matplotlib.pyplot as plt, numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

digits = load_digits()
X, y = digits.data, digits.target

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for ax, img, lab in zip(axes, digits.images, y):
    ax.imshow(img, cmap='gray_r'); ax.set_title(int(lab)); ax.axis('off')
plt.suptitle('what the machine is shown'); plt.show()

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)
clf = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
pred = clf.predict(Xte)
print(f'Test accuracy: {accuracy_score(yte, pred):.1%}')

bad = np.where(pred != yte)[0][:10]
fig, axes = plt.subplots(1, len(bad), figsize=(12, 1.8))
for ax, i in zip(np.atleast_1d(axes), bad):
    ax.imshow(Xte[i].reshape(8, 8), cmap='gray_r')
    ax.set_title(f'{yte[i]}\u2192{pred[i]}', fontsize=9); ax.axis('off')
plt.suptitle('true \u2192 guessed'); plt.show()

---
## Now break it

The accuracy above is around 96%. Here is the part that matters.

We add faint random noise to the test images — the kind of degradation any real scan has — and look at two numbers: the accuracy, **and how confident the model is when it is wrong**.

### 2 · Add noise and watch confidence stay high

**Copy this into the AI box** (click the empty cell below, then `Ctrl`+`Shift`+`Enter`, or the **Generate** button):

```text
Add faint random noise to the test images and report the accuracy again.
Then, for the noisy images it got WRONG, show me how confident it was
in its wrong answer. Plot the distribution of that confidence.
Show a few of the noisy images next to the clean originals.
```

> Watch what happens: accuracy falls substantially, and confidence barely moves. **The model has no way of telling you that it has left familiar territory.**

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
rng = np.random.default_rng(0)
Xte_noisy = np.clip(Xte + rng.normal(0, 4.0, Xte.shape), 0, 16)

pred_n = clf.predict(Xte_noisy)
conf_n = clf.predict_proba(Xte_noisy).max(axis=1)

print(f'clean accuracy : {accuracy_score(yte, pred):.1%}')
print(f'noisy accuracy : {accuracy_score(yte, pred_n):.1%}')
print()
wrong = pred_n != yte
print(f'mean confidence when WRONG on noisy images : {conf_n[wrong].mean():.1%}')
print(f'mean confidence when RIGHT on noisy images : {conf_n[~wrong].mean():.1%}')

plt.figure(figsize=(7, 3.5))
plt.hist(conf_n[wrong], bins=20, color='#B3283C')
plt.xlabel('model confidence in its WRONG answer'); plt.ylabel('count')
plt.title('It is confidently wrong'); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for j in range(8):
    axes[0, j].imshow(Xte[j].reshape(8, 8), cmap='gray_r'); axes[0, j].axis('off')
    axes[1, j].imshow(Xte_noisy[j].reshape(8, 8), cmap='gray_r'); axes[1, j].axis('off')
axes[0, 0].set_ylabel('clean'); axes[1, 0].set_ylabel('noisy')
plt.suptitle('top: what it was trained on   bottom: what it was tested on')
plt.show()

---
## The lesson

A model trained on clean data and shown slightly different data does not fail loudly. It fails **quietly and confidently**.

Now think about your own material: manuscripts photographed in different lighting, documents digitised by different institutions on different scanners, recordings made in different rooms. Every one of those is the noise in this experiment.

> **Playbook line:** when a model meets material unlike its training data, accuracy drops and confidence does not. Confidence is never evidence.